In [0]:
from pyspark.sql import functions as F

In [0]:
df_order_items = spark.read.table('global_partner_project.bronze.order_items')

In [0]:
%run /Workspace/Users/trobinuk@gmail.com/global_partners_project/silver/config_order_items

In [0]:
#print(config_rules_order_items)

In [0]:
def add_safe_casts(df,config_rules):

    result_df = df
    for column_name, rule in config_rules.items():
        typed_column_name = f"{column_name}_typed"
        target_type = rule.get('target_type','string')

        if target_type == 'datetime':
            date_format = rule.get('format','yyyy-MM-dd')
            result_df = result_df.withColumn(typed_column_name,
                                 F.try_to_timestamp(F.col(column_name)))
        elif target_type == 'string':
            result_df = result_df.withColumn(typed_column_name,
                            F.trim(F.col(column_name)))
        else:
            result_df = result_df.withColumn(typed_column_name,
                                 F.expr(f"try_cast(`{column_name}` AS {target_type})"))
    return result_df


In [0]:
typed_df_order_items = add_safe_casts(df_order_items,config_rules_order_items)
#typed_df_order_items.display()
#typed_df_order_items.printSchema()

In [0]:
def add_dq_failure_reasons(df,config_rules):

    failure_reasons = []
    for column_name,rule in config_rules.items():

        typed_column_name = f"{column_name}_typed"

        if rule.get('required',False):
            failure_reasons.append(
                F.when(F.col(column_name).isNull() |
                       (F.trim(F.col(column_name))==""),
                   F.lit(f"{column_name.upper()}_IS_NULL"))
            )

        if rule.get('target_type','string') != 'string':
            failure_reasons.append(
                F.when(F.col(column_name).isNotNull() &
                       (F.trim(F.col(column_name))!="") &
                       F.col(typed_column_name).isNull(),
                   F.lit(f"{column_name.upper()}_CASTING_FAILED"))
            )

    return df.withColumn('dq_failure_reasons',
                         F.array_compact(F.array(*failure_reasons)))


In [0]:
validated_df_items = (typed_df_order_items.transform
                (lambda df: add_dq_failure_reasons(df,config_rules_order_items))
)
validated_df_items = (validated_df_items.withColumn('dq_status',
                    F.when(F.size(F.col('dq_failure_reasons')) == 0,'VALID')
                                       .otherwise('QUARANTINED')))
#validated_df_items.display()

In [0]:
quarantined_df_items = validated_df_items.filter(F.col('dq_status')=='QUARANTINED')
quarantined_df_items.count()
#quarantined_df_items.display()

In [0]:
valid_df_items = validated_df_items.filter(F.col('dq_status')=='VALID')
print(valid_df_items.count())
#valid_df_items.display()

# Options table

In [0]:
%run /Workspace/Users/trobinuk@gmail.com/global_partners_project/silver/config_order_item_options

In [0]:
df_order_options = spark.read.table('global_partner_project.bronze.order_item_options')

In [0]:
typed_df_order_options = add_safe_casts(df_order_options,config_rules_order_item_options)
#typed_df_order_options.display()
#typed_df_order_options.printSchema()

In [0]:
validated_df_options = (typed_df_order_options.transform
                (lambda df: add_dq_failure_reasons(df,config_rules_order_item_options))
)
validated_df_options = (validated_df_options.withColumn('dq_status',
                    F.when(F.size(F.col('dq_failure_reasons')) == 0,'VALID')
                                       .otherwise('QUARANTINED')))
#validated_df_options.display()

In [0]:
quarantined_df_options = validated_df_options.filter(F.col('dq_status')=='QUARANTINED')
print(quarantined_df_options.count())
#quarantined_df_options.display()

In [0]:
valid_df_options = validated_df_options.filter(F.col('dq_status')=='VALID')
print(valid_df_options.count())
#valid_df_options.display()

# date_dim table

In [0]:
%run /Workspace/Users/trobinuk@gmail.com/global_partners_project/silver/config_date_dim

In [0]:
df_date_dim = spark.read.table('global_partner_project.bronze.date_dim')

In [0]:
typed_df_date_dim = add_safe_casts(df_date_dim,config_rules_date_dim)
#typed_df_date_dim.display()
#typed_df_date_dim.printSchema()

In [0]:
validated_df_date_dim = (typed_df_date_dim.transform
                (lambda df: add_dq_failure_reasons(df,config_rules_date_dim))
)
validated_df_date_dim = (validated_df_date_dim.withColumn('dq_status',
                    F.when(F.size(F.col('dq_failure_reasons')) == 0,'VALID')
                                       .otherwise('QUARANTINED')))
#validated_df_date_dim.display()

In [0]:
quarantined_df_date_dim = validated_df_date_dim.filter(F.col('dq_status')=='QUARANTINED')
#print(quarantined_df_date_dim.count())
#quarantined_df_date_dim.display()

In [0]:
valid_df_date_dim = validated_df_date_dim.filter(F.col('dq_status')=='VALID')
#print(valid_df_date_dim.count())
#valid_df_date_dim.display()

# Final renaming of both dataframe

In [0]:
valid_df_items = valid_df_items.select(
    F.col("app_name_typed").alias("app_name"),
    F.col("restaurant_id_typed").alias("restaurant_id"),
    F.col("creation_time_utc_typed").alias("creation_time_utc"),
    F.col("order_id_typed").alias("order_id"),
    F.col("user_id_typed").alias("user_id"),
    F.col("printed_card_number_typed").alias("printed_card_number"),
    F.col("is_loyalty_typed").alias("is_loyalty"),
    F.col("currency_typed").alias("currency"),
    F.col("lineitem_id_typed").alias("lineitem_id"),
    F.col("item_category_typed").alias("item_category"),
    F.col("item_name_typed").alias("item_name"),
    F.col("item_price_typed").alias("item_price"),
    F.col("item_quantity_typed").alias("item_quantity"),
    F.col("dq_status"),
    F.col("ingest_date")
)

In [0]:
quarantined_df_items = quarantined_df_items.select(
    F.col("app_name").alias("app_name_raw"),
    F.col("restaurant_id").alias("restaurant_id_raw"),
    F.col("creation_time_utc").alias("creation_time_utc_raw"),
    F.col("order_id").alias("order_id_raw"),
    F.col("user_id").alias("user_id_raw"),
    F.col("printed_card_number").alias("printed_card_number_raw"),
    F.col("is_loyalty").alias("is_loyalty_raw"),
    F.col("currency").alias("currency_raw"),
    F.col("lineitem_id").alias("lineitem_id_raw"),
    F.col("item_category").alias("item_category_raw"),
    F.col("item_name").alias("item_name_raw"),
    F.col("item_price").alias("item_price_raw"),
    F.col("item_quantity").alias("item_quantity_raw"),

    F.col("app_name_typed"),
    F.col("restaurant_id_typed"),
    F.col("creation_time_utc_typed"),
    F.col("order_id_typed"),
    F.col("user_id_typed"),
    F.col("printed_card_number_typed"),
    F.col("is_loyalty_typed"),
    F.col("currency_typed"),
    F.col("lineitem_id_typed"),
    F.col("item_category_typed"),
    F.col("item_name_typed"),
    F.col("item_price_typed"),
    F.col("item_quantity_typed"),

    F.col("dq_failure_reasons"),
    F.col("dq_status"),
    F.col("ingest_date"),
    F.current_timestamp().alias("quarantined_at")
)

In [0]:
valid_df_options = valid_df_options.select(
    F.col("order_id_typed").alias("order_id"),
    F.col("lineitem_id_typed").alias("lineitem_id"),
    F.col("option_group_name_typed").alias("option_group_name"),
    F.col("option_name_typed").alias("option_name"),
    F.col("option_price_typed").alias("option_price"),
    F.col("option_quantity_typed").alias("option_quantity"),
    F.col("dq_status"),
    F.col("ingest_date")
)

In [0]:
quarantined_df_options = quarantined_df_options.select(
    # Original Bronze/raw values.
    F.col("ORDER_ID").alias("order_id_raw"),
    F.col("LINEITEM_ID").alias("lineitem_id_raw"),
    F.col("OPTION_GROUP_NAME").alias("option_group_name_raw"),
    F.col("OPTION_NAME").alias("option_name_raw"),
    F.col("OPTION_PRICE").alias("option_price_raw"),
    F.col("OPTION_QUANTITY").alias("option_quantity_raw"),

    # Safe-cast values for troubleshooting.
    F.col("order_id_typed"),
    F.col("lineitem_id_typed"),
    F.col("option_group_name_typed"),
    F.col("option_name_typed"),
    F.col("option_price_typed"),
    F.col("option_quantity_typed"),

    # Quality and source/audit metadata.
    F.col("dq_failure_reasons"),
    F.col("dq_status"),
    F.col("ingest_date"),
    F.current_timestamp().alias("quarantined_at")
)

In [0]:
valid_df_date_dim = valid_df_date_dim.select(
    F.col("date_key_typed").alias("date_key"),
    F.col("year_typed").alias("year"),
    F.col("month_typed").alias("month"),
    F.col("week_typed").alias("week"),
    F.col("day_of_week_typed").alias("day_of_week"),
    F.col("is_weekend_typed").alias("is_weekend"),
    F.col("is_holiday_typed").alias("is_holiday"),
    F.col("holiday_name_typed").alias("holiday_name"),
    F.col("dq_status"),
    F.col("ingest_date")
)

In [0]:
quarantined_df_date_dim = quarantined_df_date_dim.select(
     # Original raw Bronze/source values
    F.col("date_key").alias("date_key_raw"),
    F.col("year").alias("year_raw"),
    F.col("month").alias("month_raw"),
    F.col("week").alias("week_raw"),
    F.col("day_of_week").alias("day_of_week_raw"),
    F.col("is_weekend").alias("is_weekend_raw"),
    F.col("is_holiday").alias("is_holiday_raw"),
    F.col("holiday_name").alias("holiday_name_raw"),

    # Results of safe casts/parsing
    F.col("date_key_typed"),
    F.col("year_typed"),
    F.col("month_typed"),
    F.col("week_typed"),
    F.col("day_of_week_typed"),
    F.col("is_weekend_typed"),
    F.col("is_holiday_typed"),
    F.col("holiday_name_typed"),
    # Quality and source/audit metadata.
    F.col("dq_failure_reasons"),
    F.col("dq_status"),
    F.col("ingest_date"),
    F.current_timestamp().alias("quarantined_at")
)